# DL · Encuentro 1 · Neurona, activaciones y entrenamiento

Cómo aprende una red neuronal

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [Una red transforma entradas](#scrollTo=tema-dl-redes)
2. [La activación cambia la respuesta](#scrollTo=tema-dl-activaciones)
3. [El error guía los ajustes](#scrollTo=tema-dl-entrenamiento)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · Calcular una neurona con valores dados

**Lo que vas a poder hacer:** Explicar qué entra a una neurona, qué hacen sus parámetros y cómo su salida alimenta otra capa.

### Entiende la idea

Deep Learning (aprendizaje profundo) es parte de Machine Learning (aprendizaje automático), que pertenece a la inteligencia artificial (IA). Lo «profundo» se refiere a varias capas que aprenden representaciones sucesivas de los datos. Una red neuronal transforma números mediante unidades de cálculo: los pesos multiplican entradas, el bias suma un ajuste y una activación produce la salida. Añadir capas no garantiza aprender mejor.

**Entrada:** Número que recibe la unidad para calcular su respuesta.

**Peso y bias:** El peso multiplica una entrada; el bias, o sesgo, se suma a los aportes. Son parámetros que pueden aprenderse.

**Capa y activación:** Una capa reúne unidades. La activación transforma la suma de cada una; su salida puede entrar a otra capa.

### Un caso pequeño antes del programa

**¿Cómo se transforma un par de números en una salida?**

Seguimos una neurona con dos señales abstractas, 2 y 1. Entradas, pesos y bias fueron elegidos para ver la cuenta; no son mediciones de una planta.

**Cómo leerlo:** Sigue las flechas: cada entrada se multiplica por su peso. Los aportes 1 y −1 se suman con bias 0,5. ReLU convierte una suma negativa en cero y conserva la no negativa: aquí sale 0,5.

**Para recordar:** Usar parámetros actuales es inferencia. Cambiar un control a mano explora el cálculo; todavía no entrena la neurona.

### Ahora identifica los datos de esta práctica

Reconstruir una neurona y seguir su salida hacia otra capa con parámetros dados.

1. Abre tu copia de dl-encuentro-1.ipynb en Tema 1 y ejecuta su preparación.
2. Ejecuta el cálculo de la neurona; identifica entradas, productos, bias y activación en su salida.
3. Continúa con las capas en cadena y sigue qué salida se convierte en entrada de la siguiente.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** Los parámetros están dados. Cambiar un dato y recalcular es predecir; todavía no estás entrenando. Cada celda vuelve a definir su ejemplo para poder ejecutarse sola.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### 1 · Reconstruye los aportes de una neurona

La tabla separa lo que multiplica cada entrada de la cantidad fija que añade el bias. La salida ReLU es un número intermedio, no una categoría. Esta celda tiene sus propias importaciones y datos.


In [ ]:
# Una neurona con parámetros dados: esta celda funciona por sí sola.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

entradas = np.array([2., 1.])
pesos = np.array([.5, -1.])
sesgo = .5
contribuciones = entradas * pesos
z = float(contribuciones.sum() + sesgo)
h = max(0., z)  # ReLU: conserva lo positivo y cambia lo negativo a cero.
display(pd.DataFrame({'Parte':['Entrada 1 × peso 1','Entrada 2 × peso 2','Sesgo b'],
                     'Aporte a la suma':[contribuciones[0], contribuciones[1], sesgo]}))
print('Suma z =', z, '· salida ReLU h =', h)
fig,ax = plt.subplots(figsize=(7,3))
ax.bar(['2 × 0,5','1 × (−1)','Sumar b'],[*contribuciones,sesgo],
       color=['#397ca4','#b6631f','#8455ad'])
ax.axhline(0,color='black',lw=.8)
ax.set(title='Cada entrada aporta; el bias se añade',ylabel='Aporte a la suma interna')
plt.show()
print('Entradas abstractas y parámetros inventados. Ningún valor es una probabilidad de calidad.')
print('Los pesos están fijados: ejecutar la cuenta no los entrena.')


### 2 · Sigue la salida hasta la siguiente capa

La primera salida h se convierte en entrada de otra unidad. Repetimos los parámetros del ejemplo para que puedas ejecutar esta celda por sí sola. Observa qué recibe cada capa y qué transformación entrega.


In [ ]:
# La salida de una capa se convierte en entrada de la siguiente.
import numpy as np
import pandas as pd
from IPython.display import display

# Repetimos la definición del caso para que esta celda también sea independiente.
entradas = np.array([2., 1.])
pesos = np.array([.5, -1.]); sesgo = .5
z_primera = float(entradas @ pesos + sesgo)
h = max(0., z_primera)
z_salida = 2*h - 1
p = 1/(1+np.exp(-z_salida))
display(pd.DataFrame({
    'Etapa':['Entradas','Primera suma','Activación ReLU','Segunda suma','Sigmoide final'],
    'Operación':['x₁=2; x₂=1','2×0,5 + 1×(−1) + 0,5','máximo(0, z)','2×h − 1','1/(1+exp(−z_salida))'],
    'Resultado':['2; 1',z_primera,h,z_salida,p]}))
print('h =',h,'viaja a la siguiente capa; su suma es',z_salida,'y la salida final es',p)
print('Es un fragmento mínimo de dos capas con valores dados, no una red ya entrenada.')


### Lee, comprueba y explica

**Qué debes poder comprobar:** Puedes señalar qué números son datos, cuáles son parámetros y qué transformación produce cada salida. Calcular no modifica los parámetros.

**Prueba pequeña:** Los parámetros están dados. Cambiar un dato y recalcular es predecir; todavía no estás entrenando. Cada celda vuelve a definir su ejemplo para poder ejecutarse sola.

**Pregunta:** Si la siguiente capa recibe 0,5, ¿recibe un dato nuevo o una transformación de las entradas?

**Respuesta razonada:** Recibe una transformación de las mismas entradas. Encadenar capas permite construir nuevas representaciones; no añade por sí solo una medición nueva.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-redes)

[Siguiente tema: La activación cambia la respuesta](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-activaciones)


## Tema 2 · Comparar funciones con la misma señal

**Lo que vas a poder hacer:** Comparar activaciones con la misma entrada y explicar por qué permiten respuestas diferentes.

### Entiende la idea

Una activación es una función: una regla que transforma la suma de una neurona. ReLU deja lo positivo y convierte lo negativo en cero; la sigmoide comprime a un valor entre 0 y 1; tanh, entre −1 y 1. Las activaciones no lineales permiten relaciones que varias multiplicaciones y sumas solas no representan.

**Suma z:** Resultado anterior a la activación: productos de entradas por pesos, más bias.

**ReLU:** Rectified Linear Unit: unidad lineal rectificada. Devuelve el mayor entre cero y z.

**No linealidad:** Transformación que permite que el recorrido completo vaya más allá de una única regla de multiplicar y sumar.

### Un caso pequeño antes del programa

**¿Qué cambia si la misma señal pasa por tres activaciones?**

La señal interna es −2. La enviamos por tres caminos alternativos, manteniendo exactamente la misma entrada.

**Cómo leerlo:** Lee una salida por camino: ReLU produce 0; sigmoide, aproximadamente 0,119; tanh, aproximadamente −0,964. No son tres pasos consecutivos.

**Para recordar:** La activación transforma; no aprende por sí sola. La función y la tarea determinan cómo interpretar su salida.

### Ahora identifica los datos de esta práctica

Comparar una señal y las curvas de activación, manteniendo sus entradas.

1. En dl-encuentro-1.ipynb, Tema 2, ejecuta su preparación y la comparación de activaciones.
2. Pon el mismo valor de SENAL en la celda de tabla y en la de curvas; ejecuta ambas. Son celdas independientes.
3. Lee sus curvas: entrada en horizontal, salida en vertical; relaciona cada punto con la tabla.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** Para repetir la misma comparación en tabla y curvas, usa el mismo valor de SENAL en ambas celdas. Sus importaciones y valores iniciales están incluidos.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### 1 · Compara cuatro respuestas para una misma señal

Empieza con SENAL = −2. Las filas de la tabla representan alternativas: ninguna recibe la salida de la anterior. La identidad sirve de comparación sin transformación.


In [ ]:
# Aplicamos la misma señal a todas las funciones, sin entrenar una red.
import pandas as pd
import torch
from IPython.display import display

SENAL = -2.0  # Cambia solo esta entrada y vuelve a ejecutar esta celda.
senal = torch.tensor(SENAL)
comparacion_activaciones = pd.DataFrame({
    'Función': ['Identidad', 'Sigmoide', 'tanh', 'ReLU'],
    'Entrada': [SENAL]*4,
    'Salida': [SENAL, torch.sigmoid(senal).item(), torch.tanh(senal).item(), torch.relu(senal).item()]})
display(comparacion_activaciones)
print('La entrada es una señal numérica interna, no una cantidad negativa de productos.')
print('Cambiar la señal utiliza las funciones existentes; no ajusta pesos ni entrena un modelo.')


### 2 · Ubica esa misma señal en las curvas

El punto conserva la misma entrada en los cuatro gráficos. Si cambias SENAL en la tabla, pon el mismo valor en esta celda para repetir la comparación. Las curvas no entrenan un modelo.


In [ ]:
# Cuatro alternativas para la misma señal, no una cadena de cuatro activaciones.
import numpy as np
import matplotlib.pyplot as plt

SENAL = -2.0  # Usa la misma señal que elegiste en la tabla anterior.
valores = np.linspace(-4,4,161)
funciones = [('Identidad',lambda z:z),
             ('Sigmoide',lambda z:1/(1+np.exp(-z))),
             ('tanh',np.tanh),('ReLU',lambda z:np.maximum(0,z))]
fig,axs=plt.subplots(1,4,figsize=(12,3.2))
for ax,(nombre,funcion),color in zip(axs,funciones,['#397ca4','#278071','#8455ad','#b6631f']):
    salida=float(funcion(SENAL))
    ax.plot(valores,funcion(valores),color=color)
    ax.scatter([SENAL],[salida],s=55,color=color,zorder=3)
    ax.axvline(SENAL,ls=':',color='gray');ax.axhline(0,color='gray',lw=.7)
    ax.set(title=f'{nombre}: {salida:.3f}',xlabel='Misma suma z',ylabel='Salida f(z)')
    ax.grid(alpha=.15)
plt.tight_layout();plt.show()
print('Punto = entrada elegida y su salida; curva = respuestas de esa función a otras entradas.')
print('ReLU puede superar 1; no todas las salidas se interpretan como probabilidades.')


### Lee, comprueba y explica

**Qué debes poder comprobar:** Puedes distinguir cuáles salidas admiten negativos o valores mayores que 1 y explicar por qué cambiar la función no equivale a entrenar.

**Prueba pequeña:** Para repetir la misma comparación en tabla y curvas, usa el mismo valor de SENAL en ambas celdas. Sus importaciones y valores iniciales están incluidos.

**Pregunta:** ¿ReLU(2) significa una probabilidad del 200 %?

**Respuesta razonada:** No. ReLU(2)=2 es una señal transformada, no un porcentaje. Incluso un valor entre 0 y 1 necesita un significado definido y comprobado antes de interpretarse como probabilidad.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-activaciones)

[Siguiente tema: El error guía los ajustes](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-entrenamiento)


## Tema 3 · Corregir un ajuste y entrenar una red

**Lo que vas a poder hacer:** Distinguir predecir de aprender y explicar cómo un ajuste puede reducir o aumentar la pérdida.

### Entiende la idea

Entrenar de forma supervisada usa entradas y respuestas conocidas para ajustar parámetros. Primero se predice y se calcula una pérdida, que mide el desacuerdo. La retropropagación calcula gradientes; el optimizador los usa para actualizar pesos y bias. La tasa de aprendizaje regula el tamaño del cambio. Después hay que comprobar casos nuevos.

**Etiqueta:** Respuesta conocida que permite comparar durante el entrenamiento; no se entrega al predecir un caso nuevo.

**Pérdida y gradiente:** La pérdida mide desacuerdo. El gradiente indica cómo cambia localmente esa pérdida al modificar los parámetros.

**Tasa de aprendizaje:** Número que regula el tamaño de cada actualización. Una tasa mayor no garantiza una mejora mayor.

### Un caso pequeño antes del programa

**¿Por qué un paso grande puede empeorar el ajuste?**

Para ver una sola actualización, un parámetro empieza en −2 y buscamos acercarlo a 3. Elegimos como pérdida el cuadrado de la diferencia. Las dos alternativas parten del mismo inicio.

**Cómo leerlo:** El paso pequeño se acerca al objetivo y baja la pérdida de 25 a 16. El paso excesivo lo sobrepasa y la sube a 36. El objetivo permanece quieto.

**Para recordar:** Predecir usa parámetros; aprender los cambia. Reducir la pérdida de entrenamiento no demuestra todavía que funcionen en casos nuevos.

### Ahora identifica los datos de esta práctica

Seguir un paso de ajuste y después comprobar el entrenamiento de una red pequeña.

1. En dl-encuentro-1.ipynb, Tema 3, ejecuta su preparación y el ejemplo de un parámetro.
2. Compara las dos tasas reiniciando ese ejemplo; después ejecuta la red pequeña desde su inicio.
3. Lee por separado la pérdida de entrenamiento y la evaluación de los ejemplos reservados.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** Con tasa 0,10, el primer paso llega a −1 y la pérdida baja de 25 a 16; con 1,10 llega a 9 y sube a 36. Las dos comparaciones parten del mismo inicio.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### 1 · Comprueba un solo ajuste antes de entrenar una red

Cada fila vuelve a empezar en −2. La tasa 0,10 produce −1 y pérdida 16; la tasa 1,10 produce 9 y pérdida 36. Compara estos resultados con el inicio, pérdida 25. La celda imprime la cuenta elegida, la tabla y las dos flechas.


In [ ]:
# Comparamos UN paso desde el mismo inicio; no son dos pasos consecutivos.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INICIO=-2.;OBJETIVO=3.;TASA=.10  # Cambia solo TASA y ejecuta otra vez desde el mismo inicio.
def un_paso(tasa):
    gradiente=2*(INICIO-OBJETIVO)
    nuevo=INICIO-tasa*gradiente
    return {'Tasa':tasa,'Inicio':INICIO,'Gradiente':gradiente,
            'Parámetro nuevo':nuevo,'Pérdida inicial':(INICIO-OBJETIVO)**2,
            'Pérdida nueva':(nuevo-OBJETIVO)**2}
paso_elegido=un_paso(TASA)
comparacion_pasos=pd.DataFrame([un_paso(.10),un_paso(1.10)])
display(comparacion_pasos)
print('Tu tasa elegida:',TASA,'· un paso:',INICIO,'→',paso_elegido['Parámetro nuevo'])
print('Pérdida:',paso_elegido['Pérdida inicial'],'→',paso_elegido['Pérdida nueva'])
curva=np.linspace(-3,max(10,paso_elegido['Parámetro nuevo']+1),200)
fig,ax=plt.subplots(figsize=(8,3.5))
ax.plot(curva,(curva-OBJETIVO)**2,color='#397ca4',label=f'Pérdida (parámetro − {OBJETIVO:g})²')
perdida_inicial=(INICIO-OBJETIVO)**2
ax.scatter([INICIO],[perdida_inicial],color='black',s=55,label=f'Mismo inicio: {INICIO:g}, pérdida {perdida_inicial:g}',zorder=4)
for tasa,color in [(.10,'#278071'),(1.10,'#bf3978')]:
    r=un_paso(tasa);n=r['Parámetro nuevo'];perdida=r['Pérdida nueva']
    ax.annotate('',xy=(n,perdida),xytext=(INICIO,perdida_inicial),arrowprops={'arrowstyle':'->','color':color,'lw':2})
    ax.scatter([n],[perdida],color=color,s=55,label=f'Tasa {tasa:.2f}: {n:g}, pérdida {perdida:g}')
ax.set(xlabel='Parámetro',ylabel='Pérdida',title='El tamaño del paso puede ayudar o sobrepasar el objetivo')
ax.legend(fontsize=8);plt.show()
print('Retropropagación calcula gradientes; aquí usamos la derivada de una pérdida sencilla.')
print('La red del bloque siguiente usa otros datos y otra pérdida: sus números no serán 25,16 y36.')


### Entrenar una red pequeña

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
X,y=datos_cafe(n=320);Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
escalador=StandardScaler().fit(Xtr)
xt=torch.tensor(escalador.transform(Xtr),dtype=torch.float32);yt=torch.tensor(ytr[:,None],dtype=torch.float32)
xe=torch.tensor(escalador.transform(Xte),dtype=torch.float32)
TASA=.02;TAMANO_LOTE=32;EPOCAS=40
torch.manual_seed(SEMILLA)
red=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,8),nn.ReLU(),nn.Linear(8,1))
perdida=nn.BCEWithLogitsLoss() # recibe puntuaciones; no añadir sigmoide antes
opt=torch.optim.Adam(red.parameters(),lr=TASA)
lotes=DataLoader(TensorDataset(xt,yt),batch_size=TAMANO_LOTE,shuffle=True)
historial=[]
for epoca in range(EPOCAS):
    red.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(red(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    historial.append(suma/len(xt))
red.eval()
with torch.no_grad():probs=torch.sigmoid(red(xe)).numpy().ravel()
print('Accuracy en prueba:',accuracy_score(yte,probs>=.5))
fig,axs=plt.subplots(1,2,figsize=(10,4));axs[0].plot(historial,color=AZUL);axs[0].set(title='Aprender: reducir el error',xlabel='Época',ylabel='Pérdida de entrenamiento')
xx,yy=np.meshgrid(np.linspace(0,1,80),np.linspace(0,1,80));malla=pd.DataFrame(np.c_[xx.ravel(),yy.ravel()],columns=X.columns)
with torch.no_grad():mapa=torch.sigmoid(red(torch.tensor(escalador.transform(malla),dtype=torch.float32))).numpy().reshape(xx.shape)
axs[1].contourf(xx,yy,mapa,levels=np.linspace(0,1,11),cmap='RdYlGn');axs[1].scatter(Xte.iloc[:,0],Xte.iloc[:,1],c=yte,cmap='RdYlGn',edgecolor='black',s=20);axs[1].set(title='Predecir: usar los pesos aprendidos',xlabel='Uniformidad del color',ylabel='Proporción de manchas');mostrar('09-red-entrenada')


### Después de ejecutar: tres comprobaciones pequeñas

**Datos reservados:** de los 320 casos simulados, 240 entrenan la red y 80 quedan para comprobarla. Aquí 1 significa sano y 0, defectuoso.

**Pérdida y aciertos:** el primer punto de la curva resume las pérdidas de los lotes durante la primera época; no es la pérdida antes de aprender. Compáralo con la última época sin convertirlo a porcentaje. La salida «Accuracy en prueba» sí es una proporción de aciertos entre 0 y 1: por ejemplo, 0,95 equivale al 95 %.

**Cómo leer el mapa:** el fondo representa la puntuación de la red para sano: tonos rojos para valores bajos y verdes para altos. Los puntos representan las etiquetas conocidas de los casos reservados: rojo = defectuoso; verde = sano. Compara cada punto con el fondo de su zona; un punto no es la predicción del modelo.

Si pruebas otra tasa, registra la comparación. Elegir con esa reserva la convierte en validación, no en una prueba final nueva.


### Lee, comprueba y explica

**Qué debes poder comprobar:** Explicas qué se actualizó y con qué datos. En la clasificación del cuaderno, y=1 significa sano; la pérdida y la exactitud de esa red no son las de la cuenta manual.

**Prueba pequeña:** Con tasa 0,10, el primer paso llega a −1 y la pérdida baja de 25 a 16; con 1,10 llega a 9 y sube a 36. Las dos comparaciones parten del mismo inicio.

**Pregunta:** Si cambias la tasa pero todavía no aplicas el paso, ¿ya cambió el parámetro?

**Respuesta razonada:** No. Preparaste otra actualización. El parámetro cambia al aplicar el paso con esa tasa y el gradiente calculado.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-entrenamiento)

[Siguiente tema: Tres preguntas distintas para una imagen](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-vision)


## Cierra el encuentro con una explicación

1. **Una red transforma entradas:** Usar parámetros actuales es inferencia. Cambiar un control a mano explora el cálculo; todavía no entrena la neurona.
2. **La activación cambia la respuesta:** La activación transforma; no aprende por sí sola. La función y la tarea determinan cómo interpretar su salida.
3. **El error guía los ajustes:** Predecir usa parámetros; aprender los cambia. Reducir la pérdida de entrenamiento no demuestra todavía que funcionen en casos nuevos.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
